# Project Direction, Methods, and Analysis Plan

## 1) Project Direction (Final Choice)

### Main direction
Cross-lingual transfer for Danish NER, using English labeled data as the source and Danish as the target language.

### Core research question
How well can an English-trained NER model transfer to Danish, and which lightweight transfer strategy gives the best Danish F1 under a realistic semester-project budget?

### Scope decision
Use one project idea only (cross-lingual transfer). Do not combine with a separate full data-augmentation project.

## 2) Data and Label Setup

### Source data
- English EWT NER files in this repo:
  - en_ewt-ud-train.iob2
  - en_ewt-ud-dev.iob2
  - en_ewt-ud-test-masked.iob2

### Target data
- Primary Danish dataset: DaNE (Hugging Face: alexandrainst/dane).
- Fallback: Danish WikiANN subset.

### Label schema
Use consistent BIO labels with PER/ORG/LOC and O.

## 3) Candidate Methods We May Use

## Method A: Zero-shot multilingual transfer (required baseline)
Train on English only and evaluate directly on Danish.

- Model options:
  - bert-base-multilingual-cased (mBERT)
  - xlm-roberta-base (XLM-R)
- Why include:
  - Establishes the core cross-lingual baseline.
  - Simple and reproducible.

## Method B: Few-shot continued fine-tuning on Danish (recommended main method)
Start from Method A model, then continue training with small Danish subsets.

- Suggested Danish fractions: 10%, 25%, 50%, 100%.
- Why include:
  - Answers data-efficiency question.
  - Usually provides large gains over pure zero-shot.

## Method C: Projection/translation-assisted transfer (optional extension)
Generate weak Danish labels via translation/projection and train on projected data.

- Inspired by projection and CROP-style pipelines.
- Why include:
  - Strong cross-lingual literature connection.
  - Good novelty if implementation is manageable.

## Method D: Gazetteer-enhanced transfer (optional extension)
Inject Danish entity lists (PER/ORG/LOC) from external resources.

- Use as extra features or post-processing support.
- Why include:
  - Practical low-cost boost in low-resource transfer settings.

## Recommended semester-safe method stack
1. Method A (must-have baseline)
2. Method B (main contribution)
3. One of C or D only if time allows

## 4) Proposed Experiment Matrix

1. E1: mBERT English-only -> Danish test/dev (zero-shot).
2. E2: XLM-R English-only -> Danish test/dev (zero-shot).
3. E3: Best zero-shot model + 10% Danish fine-tuning.
4. E4: Best zero-shot model + 25% Danish fine-tuning.
5. E5: Best zero-shot model + 50% Danish fine-tuning.
6. E6: Best zero-shot model + 100% Danish fine-tuning.
7. E7 (optional): Add projection or gazetteer enhancement.

## 5) Evaluation and Analysis to Perform

## Primary metrics
- Strict span-level precision/recall/F1 (CoNLL style).
- Use repo evaluators:
  - conlleval.pl
  - span_f1.py

## Secondary metrics
- Per-class F1 (PER, ORG, LOC).
- Unlabeled span F1.
- Loose overlap F1.

## Required analyses
1. Learning curve analysis:
- Plot F1 vs Danish fine-tuning fraction (10/25/50/100).

2. Error type analysis:
- Boundary errors vs label errors.
- Cases where span is correct but entity type is wrong.

3. Per-class behavior:
- Which entities transfer best from English to Danish (PER/ORG/LOC)?

4. Qualitative error analysis:
- 20 to 50 sampled errors with short categories (name variants, capitalization, multi-token spans, etc.).

5. Ablation (at least one):
- mBERT vs XLM-R under same training setup.
- Optional second ablation: with/without optional enhancement (C or D).

## 6) Reproducibility Requirements

- Fixed random seeds.
- Fixed train/dev/test split definitions.
- Keep prediction output format aligned with scorer scripts.
- Document exact hyperparameters and model checkpoints.

## 7) Risks and Mitigations

1. Label/schema mismatch between datasets.
- Mitigation: explicit mapping checks before training.

2. Domain mismatch between English and Danish corpora.
- Mitigation: report zero-shot gap and rely on few-shot adaptation curve.

3. Time/compute limits.
- Mitigation: prioritize A+B and one clean analysis package before optional methods.

## 8) Deliverable Priorities

1. Strong, reproducible baseline (A).
2. Clear improvement story via few-shot transfer (B).
3. Solid analysis section with tables/plots and error discussion.
4. Optional extension only if baseline and analysis are complete.



# How to Reuse sequence_labeling.ipynb for This Project

Notebook reviewed:
- /Users/erikaymerich/Downloads/sequence_labeling.ipynb

## What is directly reusable
1. Tokenization + label alignment logic (`tokenize_and_align_labels`).
2. Data collator and PyTorch DataLoader setup.
3. Training loop skeleton (TODO B block).
4. seqeval metric computation structure (`get_labels`, `compute_metrics`).

## What must be changed for your project
1. Replace dataset loader:
- Current notebook loads `conll2003` via `load_dataset`.
- Your project uses local IOB2 files and later Danish dataset files.

2. Remove HPC shell cells:
- `ssh`, `srun`, `conda create` cells are shell commands, not notebook Python code for local execution.

3. Match project evaluation format:
- Keep sentence boundaries and token order.
- Write predictions in same CoNLL-like format expected by `span_f1.py` and `conlleval.pl`.

4. Keep label set consistent:
- English source labels are PER/ORG/LOC in BIO tags.
- Ensure Danish labels map to exactly the same scheme before training/evaluation.

## Minimum baseline implementation order
1. Build local data reader for IOB2 files.
2. Build Hugging Face DatasetDict from local splits.
3. Reuse tokenizer/label alignment function.
4. Implement TODO B training loop.
5. Implement TODO C `get_labels` function.
6. Run dev evaluation with seqeval.
7. Export prediction file and evaluate with `span_f1.py` + `conlleval.pl`.

## Recommended first baseline config
- Model: `bert-base-multilingual-cased` (for cross-lingual path)
- Learning rate: 2e-5
- Epochs: 3 to 5
- Batch size: 8 or 16
- Max length: 128
- Seed: 42

## Cross-lingual experiment sequence
1. Train on English only -> evaluate Danish (zero-shot).
2. Continue fine-tuning with 10/25/50/100% Danish train splits.
3. Compare against Danish-only training.


# Cross-Lingual Transfer for Danish NER from English Supervision

## Title
Cross-Lingual Transfer for Danish Named Entity Recognition with English-Supervised Transformers

## Team
[Add names and emails]

## Motivation and Context
Named Entity Recognition (NER) systems perform well in high-resource settings, but many languages have less annotated data. Danish is lower-resource than English for NER, and cross-lingual transfer with multilingual encoders is a practical way to reduce annotation needs. Our project studies how far we can transfer an English-trained NER model to Danish, and how much Danish supervision is needed to close the gap.

## Research Question
How effective is cross-lingual transfer from English to Danish for NER with PER/ORG/LOC labels, and what is the gain from adding limited Danish fine-tuning data?

## Hypotheses
1. Zero-shot transfer (train on English only, test on Danish) will outperform trivial baselines but remain clearly below Danish-supervised performance.
2. Continued fine-tuning with a small Danish subset will significantly improve Danish F1 over zero-shot.
3. XLM-R-base will outperform mBERT in zero-shot and few-shot settings.

## Data
### Source language (training)
- English Universal NER EWT files already in this repo:
  - en_ewt-ud-train.iob2
  - en_ewt-ud-dev.iob2
  - en_ewt-ud-test-masked.iob2
- Labels in source data: PER, ORG, LOC (BIO tags).

### Target language (Danish)
Primary dataset:
- DaNE (Danish NER resource): https://aclanthology.org/2020.lrec-1.565/
- Hugging Face dataset ID: alexandrainst/dane
- License reported by dataset card: CC BY-SA 4.0

Fallback dataset if access issues occur:
- WikiANN Danish (da) via multilingual WikiANN resources
- Mentioned in DaNLP dataset docs as available for Danish and compatible PER/ORG/LOC setting.

## Method
### Baseline model
- Token-classification fine-tuning with multilingual encoder.
- Initial baseline: mBERT (bert-base-multilingual-cased).
- Stronger variant: XLM-R-base.

### Transfer setups
1. Zero-shot:
- Train on English only.
- Evaluate directly on Danish test/dev.

2. Few-shot transfer:
- Train on English, then continue fine-tuning on 10%, 25%, 50%, and 100% Danish train split.
- Evaluate on fixed Danish dev/test.

3. Monolingual Danish reference (upper-bound within project scope):
- Train and evaluate on Danish only using same architecture and protocol.

### Evaluation
- Primary metric: strict span-level F1 (CoNLL style).
- Secondary metrics: precision, recall, per-label F1 (PER/ORG/LOC).
- Use project evaluators where possible:
  - conlleval.pl
  - span_f1.py

## Analysis Plan
1. Learning-curve analysis over Danish fine-tuning fraction (10/25/50/100%).
2. Error analysis:
- Boundary errors vs label errors.
- Per-class performance (PER/ORG/LOC).
- Typical confusion patterns and short qualitative examples.
3. Ablation:
- Compare mBERT vs XLM-R under identical training settings.

## Expected Contribution
- A reproducible cross-lingual NER pipeline from English to Danish.
- Clear estimate of data-efficiency: how much Danish data is needed to approach monolingual performance.
- Practical guidance for low-resource NER in related Germanic languages.

## Feasibility and Timeline
Week 1:
- Finalize dataset loading and label mapping.
- Run English baseline and zero-shot Danish evaluation.

Week 2:
- Run few-shot Danish transfer experiments.
- Start learning-curve and per-class analysis.

Week 3:
- Complete ablation (mBERT vs XLM-R).
- Finalize qualitative error analysis and write report.

## Risks and Mitigations
- Risk: Dataset format mismatch (IOB2 vs other format).
  - Mitigation: Build robust converter to the project evaluation format and validate on sample sentences.
- Risk: Domain mismatch between English and Danish corpora.
  - Mitigation: Report and analyze errors; include few-shot adaptation to reduce mismatch.
- Risk: Compute limits.
  - Mitigation: Use base-size models and fixed hyperparameter budget.

## Core References
- Hvingelby et al. 2020, DaNE: https://aclanthology.org/2020.lrec-1.565/
- Conneau et al. 2020, XLM-R: https://aclanthology.org/2020.acl-main.747/
- Pan et al. 2017, WikiANN: https://aclanthology.org/P17-1178/
